# 05 RAGAS Evaluation

생성 완료 CSV 3개를 입력으로 받아 RAGAS 평가만 수행합니다. 답변 생성과 retrieval은 재실행하지 않습니다.

## Git Clone

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/songhahyun/finance-terms-rag-chatbot.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path('/content/finance-terms-rag-chatbot')

In [ ]:
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print('cwd =', Path.cwd())

In [ ]:
# Python deps
!pip install -q -U pip
!pip install -q -r requirements.txt

## Fetch API Keys

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "OPENAI_API_KEY",
    "NCP_APIGW_API_KEY",
    "CLOVASTUDIO_API_KEY",
    "HF_TOKEN",
    "WANDB_API_KEY",
]

missing = []

for name in REQUIRED_SECRETS:
    value = userdata.get(name)
    if value:
        os.environ[name] = value
    else:
        missing.append(name)

if missing:
    raise RuntimeError(
        "Colab Secrets에 다음 값을 추가하고 Notebook access를 켜세요: "
        + ", ".join(missing)
    )

## Run evaluation

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
from src.common.config import get_settings
from src.evaluation.ragas_pipeline import run_ragas_evaluations

settings = get_settings()

GENERATION_OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "generation_001_baseline"
RAGAS_OUTPUT_DIR = ROOT / "data" / "eval" / "outputs" / "ragas_001_baseline"
RAGAS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_JSON_PATH = settings.default_chunk_json_path
JUDGE_MODEL = "gpt-4o-mini"
JUDGE_EMBEDDING_MODEL = "multilingual-e5-large"
MAX_ROWS = 5  # Set to None for the full evaluation.

USE_WEAVE = True
WEAVE_PROJECT = "finance-terms-rag-evaluation"
WEAVE_EXPERIMENT_GROUP = "generation_001_baseline"
WEAVE_LOG_CONTEXTS = True
WEAVE_PRINT_CALL_LINK = False

TARGET_CSV_FILES = [
    "dense_clova_bge-m3.csv",
    "hybrid_clova_bge-m3.csv",
    "hybrid_openai_text-embedding-3-small.csv",
]

print("Generation output dir:", GENERATION_OUTPUT_DIR)
print("RAGAS output dir:", RAGAS_OUTPUT_DIR)
print("Chunk JSON:", CHUNK_JSON_PATH)
print("Judge model:", JUDGE_MODEL)
print("Judge embedding model:", JUDGE_EMBEDDING_MODEL)
print("Max rows:", MAX_ROWS)
print("Use Weave:", USE_WEAVE)
print("Weave project:", WEAVE_PROJECT)
print("Weave experiment group:", WEAVE_EXPERIMENT_GROUP)

In [ ]:
missing_files = [name for name in TARGET_CSV_FILES if not (GENERATION_OUTPUT_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing generation CSV files: {missing_files}")

generated_csv_paths = [GENERATION_OUTPUT_DIR / name for name in TARGET_CSV_FILES]
combined_summary_path = RAGAS_OUTPUT_DIR / "ragas_experiment_summary.csv"

detail_outputs, summary_df = run_ragas_evaluations(
    generated_csv_paths=generated_csv_paths,
    chunk_json_path=CHUNK_JSON_PATH,
    output_dir=RAGAS_OUTPUT_DIR,
    output_summary_path=combined_summary_path,
    judge_model=JUDGE_MODEL,
    judge_embedding_model=JUDGE_EMBEDDING_MODEL,
    max_rows=MAX_ROWS,
    use_weave=USE_WEAVE,
    weave_project=WEAVE_PROJECT,
    weave_experiment_group=WEAVE_EXPERIMENT_GROUP,
    weave_log_contexts=WEAVE_LOG_CONTEXTS,
    weave_print_call_link=WEAVE_PRINT_CALL_LINK,
)

print("Saved RAGAS output files:")
for name, path in detail_outputs.items():
    print(f"- {name}: {path}")
print(f"- combined summary: {combined_summary_path}")

summary_df